# Tier 1 Balance Scaler

Scales numeric values in **one or more XML files** by configurable per-node-type factors.

## Workflow
1. **Cell 1** – run once; installs deps and sets up all helpers.
2. **Cell 2** – set `TARGET_FILES` to the list of XMLs you want to rebalance, then run to discover every numeric node across all of them.
3. **Cell 3** – for each discovered node type, choose a scale factor or ignore it.
4. **Cell 4** – dry-run: preview all changes without writing any files.
5. **Cell 5** – set `APPLY = True` then run to write changes (backups created automatically).
6. **Cell 6** – re-run to verify all changes were applied.

> **To process a different batch:** edit `TARGET_FILES` in Cell 2 and re-run Cells 2–6.


In [1]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────────
import subprocess, sys

for _pkg in ('lxml', 'pandas'):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', _pkg])

from pathlib import Path
import lxml.etree as ET
import pandas as pd
import shutil, re

# Notebook lives in Scripts/; VS Code sets cwd to the notebook's directory.
SCRIPT_DIR = Path().resolve()
MOD_ROOT   = SCRIPT_DIR.parent / '1.6'

# ── Pure helpers ──────────────────────────────────────────────────────────────
_NUMERIC_RE = re.compile(r'^-?\d+(?:\.\d+)?$')
_RANGE_RE   = re.compile(r'^-?\d+(?:\.\d+)?~-?\d+(?:\.\d+)?$')


def _is_numeric_text(text: str) -> bool:
    t = text.strip()
    return bool(_NUMERIC_RE.match(t) or _RANGE_RE.match(t))


def _find_def_name(el) -> str:
    """Walk up the lxml tree to find the nearest defName or Name attribute."""
    node = el
    while node is not None:
        for attr in ('Name', 'defName'):
            v = node.get(attr)
            if v:
                return v
        dn = node.find('defName')
        if dn is not None and dn.text:
            return dn.text.strip()
        node = node.getparent()
    return '?'


def scale_value(text: str, factor: float) -> str:
    """Scale a numeric XML text value.  Integers stay integers; floats stay 2 dp."""
    def _scale_part(part: str) -> str:
        v      = float(part)
        scaled = v * factor
        return f'{scaled:.2f}' if '.' in part else str(int(round(scaled)))
    text = text.strip()
    if '~' in text:
        lo, hi = text.split('~', 1)
        return f'{_scale_part(lo)}~{_scale_part(hi)}'
    return _scale_part(text)


def resolve_factor(parent_tag: str, child_tag: str, mapping: dict):
    """Exact 'parent/child' first, then wildcard 'parent/*'.  None if no match."""
    return mapping.get(f'{parent_tag}/{child_tag}', mapping.get(f'{parent_tag}/*'))


def backup_file(path: Path) -> bool:
    """Copy path into Scripts/backup/<original relative path>.  Returns True if backed up."""
    bak = SCRIPT_DIR / 'backup' / path.relative_to(MOD_ROOT.parent)
    bak.parent.mkdir(parents=True, exist_ok=True)
    if bak.exists() and bak.stat().st_mtime >= path.stat().st_mtime:
        return False
    shutil.copy2(str(path), str(bak))
    return True


print(f'MOD_ROOT : {MOD_ROOT}')
print('Setup complete.')


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


MOD_ROOT : /mnt/c/Program Files (x86)/Steam/steamapps/common/RimWorld/Mods/BeyondOurReach/1.6
Setup complete.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# ── Cell 2: File Selection & Discovery ───────────────────────────────────────
# ┌─────────────────────────────────────────────────────────────────────────────┐
# │  Edit TARGET_FILES to the XMLs you want to rebalance, then run this cell.  │
# └─────────────────────────────────────────────────────────────────────────────┘

_RS   = MOD_ROOT / 'Mods/Rimsenal/Patches'
_RAV  = MOD_ROOT / 'Mods/rimsenalavp/Patches'
_RF   = MOD_ROOT / 'Mods/RimsenalFederation/Patches'
_RA   = MOD_ROOT / 'Mods/RoyalArsenal/Patches'
_SC   = MOD_ROOT / 'Mods/scarmory/Patches'
_VAE  = MOD_ROOT / 'Mods/vae/Patches'
_ROY  = MOD_ROOT / 'Mods/Royalty/Patches'
_BP2  = MOD_ROOT / 'Common/Defs/BodyPartDefs/Tier2'
_BLD2 = MOD_ROOT / 'Common/Defs/ThingDefs_Buildings/tier2'
_ITM2 = MOD_ROOT / 'Common/Defs/ThingDefs_Items/Tier2'
_BIO  = MOD_ROOT / 'Mods/Biotech/Patches'
_EPOE = MOD_ROOT / 'Mods/EPOE/Patches'

TARGET_FILES = [
    # ── Common/Defs/BodyPartDefs/Tier2 ───────────────────────────────────────
    _BP2 / 'BOR_CombatStatImplants.xml',
    _BP2 / 'BOR_StatBoostImplants.xml',
    _BP2 / 'BOR_StatBoostImplants_Expanded.xml',
    _BP2 / 'Bodyparts.xml',
    # ── Common/Defs/ThingDefs_Buildings/tier2 ────────────────────────────────
    _BLD2 / 'Buildings_Joy.xml',
    _BLD2 / 'Buildings_Misc.xml',
    _BLD2 / 'Buildings_Power.xml',
    _BLD2 / 'Buildings_Production.xml',
    _BLD2 / 'Buildings_Turret.xml',
    # ── Common/Defs/ThingDefs_Items/Tier2 ────────────────────────────────────
    _ITM2 / 'Armor/BOR_StatBoostApparel.xml',
    _ITM2 / 'Armor/apparel_armor.xml',
    _ITM2 / 'Armor/apparel_headgear.xml',
    _ITM2 / 'Drugs/Canter.xml',
    _ITM2 / 'Drugs/Castra.xml',
    _ITM2 / 'Drugs/Sleepquitin.xml',
    _ITM2 / 'Drugs/Sunlight.xml',
    _ITM2 / 'Resources/quasaralloy.xml',
    _ITM2 / 'Resources/quasaralloyweave.xml',
    _ITM2 / 'Various/Tier2_Apparel_Belts.xml',
    _ITM2 / 'Various/medical.xml',
    _ITM2 / 'WeaponRanged/MeleeUltratech.xml',
    _ITM2 / 'WeaponRanged/RangedUltratech.xml',
    # ── Mods/Biotech/Patches ─────────────────────────────────────────────────
    _BIO / 'Tier2Mechs/MechaGuns/RangedMechanoid_Light.xml',
    _BIO / 'Tier2Mechs/MechaGuns/RangedMechanoid_Medium.xml',
    _BIO / 'Tier2Mechs/RaceKinds/Races_Mechanoids_Heavy.xml',
    _BIO / 'Tier2Mechs/RaceKinds/Races_Mechanoids_Light.xml',
    _BIO / 'Tier2Mechs/RaceKinds/Races_Mechanoids_Medium.xml',
    _BIO / 'Tier2Mechs/Recipes/Recipes_MechGestator_Heavy.xml',
    _BIO / 'Tier2Mechs/Recipes/Recipes_MechGestator_Light.xml',
    _BIO / 'Tier2Mechs/Recipes/Recipes_MechGestator_Medium.xml',
    _BIO / 'tier2hediffs.xml',
    # ── Mods/EPOE/Patches ────────────────────────────────────────────────────
    _EPOE / 'hediffdefs_bionicsribstier2.xml',
]

# Tags that are never worth scaling
_SKIP_TAGS = {'workAmount', 'workToBuild', "WorkToMake", "generateCommonality", "displayPriority", "equippedAngleOffset"}
# If any ancestor has one of these tags, skip the node entirely
_SKIP_ANCESTOR_TAGS = {'graphicData', 'skillRequirements'}

# ── Scan all files ────────────────────────────────────────────────────────────
_records = []
_missing = []

for _tf in [Path(f) for f in TARGET_FILES]:
    if not _tf.exists():
        _missing.append(str(_tf))
        continue
    print(f'Scanning: {_tf.relative_to(MOD_ROOT.parent)}')
    try:
        _tree = ET.parse(str(_tf))
    except ET.XMLSyntaxError as _exc:
        print(f'  ⚠  Parse error: {_exc}')
        continue
    for _el in _tree.getroot().iter():
        if not isinstance(_el.tag, str):
            continue
        if _el.tag in _SKIP_TAGS:
            continue
        _text = (_el.text or '').strip()
        if not _is_numeric_text(_text):
            continue
        _parent = _el.getparent()
        if _parent is None:
            continue
        # Skip nodes inside ignored ancestor containers
        if _SKIP_ANCESTOR_TAGS.intersection(
                _a.tag for _a in _el.iterancestors() if isinstance(_a.tag, str)):
            continue
        _records.append({
            'parent_tag': _parent.tag,
            'child_tag':  _el.tag,
            'path':       f'{_parent.tag}/{_el.tag}',
            'value':      _text,
            'has_neg':    _text.startswith('-') or (
                              '~' in _text and any(p.startswith('-') for p in _text.split('~'))),
            'def_name':   _find_def_name(_el),
        })

if _missing:
    print(f'\n⚠  {len(_missing)} file(s) not found:')
    for _m in _missing:
        print(f'  {_m}')

if not _records:
    print('\nNo numeric nodes found across these files.')
    _discovered_paths = []
else:
    _df = pd.DataFrame(_records)
    _disc_df = (
        _df.groupby(['path', 'parent_tag', 'child_tag'])
        .agg(
            count    = ('value', 'count'),
            examples = ('value', lambda x: ', '.join(sorted(set(x))[:6])),
            has_neg  = ('has_neg', 'any'),
            defs     = ('def_name', lambda x: ', '.join(sorted(set(x))[:4])),
        )
        .reset_index()
        .sort_values('path')
    )
    # Drop node types that appear only once — not worth batch-scaling
    _disc_df = _disc_df[_disc_df['count'] > 1].reset_index(drop=True)
    _discovered_paths = list(_disc_df['path'])

    pd.set_option('display.max_rows', 300)
    pd.set_option('display.max_colwidth', 70)
    _n_files = len([f for f in TARGET_FILES if Path(f).exists()])
    print(f'\nFound {len(_disc_df)} unique numeric node type(s) across {_n_files} file(s):')
    display(_disc_df[['path', 'count', 'examples', 'has_neg', 'defs']])
    print(f'\n→ Run Cell 3 to configure which of these {len(_discovered_paths)} paths to scale.')


Scanning: 1.6/Common/Defs/BodyPartDefs/Tier2/BOR_CombatStatImplants.xml
Scanning: 1.6/Common/Defs/BodyPartDefs/Tier2/BOR_StatBoostImplants.xml
Scanning: 1.6/Common/Defs/BodyPartDefs/Tier2/BOR_StatBoostImplants_Expanded.xml
Scanning: 1.6/Common/Defs/BodyPartDefs/Tier2/Bodyparts.xml
Scanning: 1.6/Common/Defs/ThingDefs_Buildings/tier2/Buildings_Joy.xml
Scanning: 1.6/Common/Defs/ThingDefs_Buildings/tier2/Buildings_Misc.xml
Scanning: 1.6/Common/Defs/ThingDefs_Buildings/tier2/Buildings_Power.xml
Scanning: 1.6/Common/Defs/ThingDefs_Buildings/tier2/Buildings_Production.xml
Scanning: 1.6/Common/Defs/ThingDefs_Buildings/tier2/Buildings_Turret.xml
Scanning: 1.6/Common/Defs/ThingDefs_Items/Tier2/Armor/BOR_StatBoostApparel.xml
Scanning: 1.6/Common/Defs/ThingDefs_Items/Tier2/Armor/apparel_armor.xml
Scanning: 1.6/Common/Defs/ThingDefs_Items/Tier2/Armor/apparel_headgear.xml
Scanning: 1.6/Common/Defs/ThingDefs_Items/Tier2/Drugs/Canter.xml
Scanning: 1.6/Common/Defs/ThingDefs_Items/Tier2/Drugs/Castra.xml

,path,count,examples,has_neg,defs
0,AmmoUser/magazineSize,3,"10, 30, 8",False,"BOR_Quasar_Gun_MiniFlameblaster, BOR_Quasar_Gun_MiniShotgun, BOR_Q..."
1,AmmoUser/reloadTime,3,"4, 4.9",False,"BOR_Quasar_Gun_MiniFlameblaster, BOR_Quasar_Gun_MiniShotgun, BOR_Q..."
2,HediffDef/initialSeverity,5,1,False,"ControlSublinkImplantQuasar, MechFormfeederImplantQuasar, RemoteRe..."
3,HediffDef/maxSeverity,9,"1.0, 3, 6",False,"BOR_CanterHigh, BOR_CastraHigh, BOR_SleepquitinHigh, BOR_SunlightHigh"
4,HediffDef/minSeverity,5,0,False,"ControlSublinkImplantQuasar, MechFormfeederImplantQuasar, RemoteRe..."
5,PawnKindDef/combatPower,8,"10, 150, 250, 400, 45, 600",False,"BOR_Quasar_Mech_Apocriton, BOR_Quasar_Mech_Legionary, BOR_Quasar_M..."
6,PawnKindDef/controlGroupPortraitZoom,4,"0.8, 1, 1.2, 1.8",False,"BOR_Quasar_Mech_Legionary, BOR_Quasar_Mech_Tunneler, HeavyQuasarMe..."
7,PawnKindDef/techHediffsChance,6,1,False,"BOR_Quasar_Mech_Apocriton, BOR_Quasar_Mech_Legionary, BOR_Quasar_M..."
8,PawnKindDef/techHediffsMoney,6,9999~9999,False,"BOR_Quasar_Mech_Apocriton, BOR_Quasar_Mech_Legionary, BOR_Quasar_M..."
9,PawnKindDef/weaponMoney,6,9999~9999,False,"BOR_Quasar_Mech_Apocriton, BOR_Quasar_Mech_Legionary, BOR_Quasar_M..."



→ Run Cell 3 to configure which of these 172 paths to scale.


In [3]:
# ── Cell 3: Scaling Config (Interactive) ─────────────────────────────────────
# For each discovered node type, choose:
#   Enter        → scale at STAT_FACTOR (or cached value in review mode)
#   a number     → scale at that custom factor
#   i / ignore   → exclude from scaling

import json
import datetime

STAT_FACTOR: float = 0.83334
COST_FACTOR: float = STAT_FACTOR   # ← adjust for separate cost scaling

_CACHE_FILE = SCRIPT_DIR / 'node_scaling_cache.json'
NODE_SCALING: dict = {}

if not _discovered_paths:
    print('No paths to configure — re-run Cell 2 first.')
else:
    # ── Load cache if available ───────────────────────────────────────────────
    _cache      = {}
    _cache_mode = 'fresh'

    if _CACHE_FILE.exists():
        _cache       = json.loads(_CACHE_FILE.read_text())
        _cache_hits  = sum(1 for p in _discovered_paths if p in _cache)
        _cache_new   = sum(1 for p in _discovered_paths if p not in _cache)
        _ts_str      = datetime.datetime.fromtimestamp(_CACHE_FILE.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
        _load_prompt = (
            f'Cache found ({_ts_str}  |  {len(_cache)} stored entries'
            f'  |  {_cache_hits}/{len(_discovered_paths)} match current paths'
            + (f'  |  {_cache_new} new' if _cache_new else '') + ')\n'
            f'  [Enter] = apply all  /  r = review each  /  n = start fresh: '
        )
        _ans = input(_load_prompt).strip().lower()
        if _ans == 'n':
            _cache_mode = 'fresh'
            print('  → Starting fresh.\n')
        elif _ans == 'r':
            _cache_mode = 'review'
            print('  → Reviewing with cache as defaults.\n')
        else:
            _cache_mode = 'apply_all'
            print('  → Applying all from cache.\n')

    # ── Apply-all mode ────────────────────────────────────────────────────────
    if _cache_mode == 'apply_all':
        _skipped = []
        for _path in _discovered_paths:
            if _path in _cache:
                NODE_SCALING[_path] = _cache[_path]
                print(f'  {_path:<52} ×{_cache[_path]}')
            else:
                _skipped.append(_path)
        if _skipped:
            print(f'\n  ⚠  {len(_skipped)} path(s) not in cache (re-run in review mode to configure):')
            for _p in _skipped:
                print(f'    {_p}')

    # ── Interactive mode (fresh or review) ────────────────────────────────────
    else:
        _info  = _disc_df.set_index('path')[['count', 'examples', 'has_neg']].to_dict('index')
        _total = len(_discovered_paths)
        if _cache_mode == 'review':
            print(f'Configuring {_total} node type(s).  Enter = cached value (or ×{STAT_FACTOR} if new) | number = custom | i = ignore\n')
        else:
            print(f'Configuring {_total} node type(s).  Enter = ×{STAT_FACTOR} | number = custom | i = ignore\n')

        for _i, _path in enumerate(_discovered_paths, 1):
            _d          = _info.get(_path, {})
            _neg        = ' [has negatives]' if _d.get('has_neg') else ''
            _ex         = _d.get('examples', '')
            _cnt        = _d.get('count', '?')
            _cached_val = _cache.get(_path)
            if _cache_mode == 'review' and _cached_val is not None:
                _default_hint = f'×{_cached_val} [Enter=cached]'
            else:
                _default_hint = f'×{STAT_FACTOR} [Enter]'
            _prompt = (
                f'[{_i}/{_total}] {_path}  ({_cnt}x  e.g. {_ex}){_neg}\n'
                f'  {_default_hint} / custom factor / i to ignore: '
            )
            while True:
                _ans = input(_prompt).strip().lower()
                if _ans in ('i', 'ignore'):
                    print('  → ignored\n')
                    break
                elif _ans == '':
                    _f = _cached_val if (_cache_mode == 'review' and _cached_val is not None) else STAT_FACTOR
                    NODE_SCALING[_path] = _f
                    print(f'  → ×{_f}\n')
                    break
                else:
                    try:
                        _f = float(_ans)
                        NODE_SCALING[_path] = _f
                        print(f'  → ×{_f}\n')
                        break
                    except ValueError:
                        _prompt = '  Invalid — enter a number, or "i" to ignore: '

    # ── Save cache (merge: existing entries kept, current run updates/adds) ───
    _merged = {**_cache, **NODE_SCALING}
    _CACHE_FILE.write_text(json.dumps(_merged, indent=2))
    _added   = sum(1 for k in NODE_SCALING if k not in _cache)
    _updated = sum(1 for k in NODE_SCALING if k in _cache and _cache[k] != NODE_SCALING[k])
    print(f'\nCache saved → {_CACHE_FILE.name}  ({len(_merged)} total entries', end='')
    if _added or _updated:
        print(f', {_added} added, {_updated} updated)', end='')
    print(')')
    print(f'Done. NODE_SCALING: {len(NODE_SCALING)} active entr{"y" if len(NODE_SCALING)==1 else "ies"}\n')
    for _k, _v in sorted(NODE_SCALING.items()):
        print(f'  {_k:<52} ×{_v}')


  → Reviewing with cache as defaults.

Configuring 172 node type(s).  Enter = cached value (or ×0.83334 if new) | number = custom | i = ignore

  → ×1.0

  → ×1.1

  → ×1.0

  → ×1.0

  → ×1.0

  → ×0.83334

  → ×1.0

  → ×1.0

  → ×1.0

  → ×1.0

  → ×1.0

  → ×0.83334

  → ×1.1

  → ×1.1

  → ×1.0

  → ×1.1

  → ×1.1

  → ×1.0

  → ×1.0

  → ×0.83334

  → ×1.0

  → ×1.0

  → ×1.0

  → ×1.0

  → ×1.0

  → ×1.0

  → ×1.0

  → ×1.0

  → ×0.83334

  → ×1.0

  → ×1.0

  → ×0.83334

  → ×1.1

  → ×1.1

  → ×1.1

  → ×1.1

  → ×0.83334

  → ×1.1

  → ×1.1

  → ×1.1

  → ×1.0

  → ×1.0

  → ×1.1

  → ×0.83334

  → ×0.83334

  → ×0.83334

  → ×0.83334

  → ×0.83334

  → ×1.0

  → ×1.0

  → ×1.0

  → ×1.0

  → ×0.83334

  → ×0.83334

  → ×0.83334

  → ×0.83334

  → ×0.83334

  → ×1.0

  → ×0.83334

  → ×0.83334

  → ×0.83334

  → ×0.83334

  → ×0.83334

  → ×0.83334

  → ×0.83334

  → ×1.0

  → ×1.0

  → ×1.1

  → ×1.0

  → ×0.83334

  → ×0.83334

  → ×0.83334

  → ×1.0

  → ×1.0

  → ×1.0

  

In [4]:
# ── Cell 4: Dry-Run Preview ──────────────────────────────────────────────────
# Computes all changes without writing any files.


def preview_changes(files: list, mapping: dict) -> pd.DataFrame:
    rows = []
    for fpath in files:
        try:
            tree = ET.parse(str(fpath))
        except ET.XMLSyntaxError as exc:
            print(f'⚠  Parse error: {fpath.name}: {exc}')
            continue
        rel = str(fpath.relative_to(MOD_ROOT.parent))
        for el in tree.getroot().iter():
            if not isinstance(el.tag, str):
                continue
            text = (el.text or '').strip()
            if not _is_numeric_text(text):
                continue
            parent = el.getparent()
            if parent is None:
                continue
            factor = resolve_factor(parent.tag, el.tag, mapping)
            if factor is None:
                continue
            new_text = scale_value(text, factor)
            if new_text == text:
                continue
            rows.append({
                'file':   rel,
                'def':    _find_def_name(el),
                'path':   f'{parent.tag}/{el.tag}',
                'old':    text,
                'new':    new_text,
                'factor': factor,
            })
    cols = ['file', 'def', 'path', 'old', 'new', 'factor']
    return pd.DataFrame(rows, columns=cols) if rows else pd.DataFrame(columns=cols)


_target_paths = [Path(f) for f in TARGET_FILES]
_preview_df = preview_changes(_target_paths, NODE_SCALING)
print(f'Dry-run: {len(_preview_df)} node(s) would change across {len(_target_paths)} file(s)\n')
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_colwidth', 90)
display(_preview_df)


Dry-run: 908 node(s) would change across 32 file(s)



,file,def,path,old,new,factor
0,1.6/Common/Defs/BodyPartDefs/Tier2/BOR_CombatStatImplants.xml,BOR_QuasarDuelflowLattice,costList/BOR_QuasarAlloy,10,11,1.10000
1,1.6/Common/Defs/BodyPartDefs/Tier2/BOR_CombatStatImplants.xml,BOR_QuasarDuelflowLattice,statBases/Mass,0.17,0.19,1.10000
2,1.6/Common/Defs/BodyPartDefs/Tier2/BOR_CombatStatImplants.xml,BOR_QuasarGraviticStrikeGovernor,costList/BOR_QuasarAlloy,10,11,1.10000
3,1.6/Common/Defs/BodyPartDefs/Tier2/BOR_CombatStatImplants.xml,BOR_QuasarGraviticStrikeGovernor,statBases/Mass,0.17,0.19,1.10000
4,1.6/Common/Defs/BodyPartDefs/Tier2/BOR_CombatStatImplants.xml,BOR_QuasarReflexOverclockSpindle,costList/BOR_QuasarAlloy,10,11,1.10000
...,...,...,...,...,...,...
903,1.6/Mods/EPOE/Patches/hediffdefs_bionicsribstier2.xml,RespirationRibQuasar,statBases/Mass,0.6,0.66,1.10000
904,1.6/Mods/EPOE/Patches/hediffdefs_bionicsribstier2.xml,WakeUpRibQuasar,statOffsets/MentalBreakThreshold,0.05,0.04,0.83334
905,1.6/Mods/EPOE/Patches/hediffdefs_bionicsribstier2.xml,WakeUpRibQuasar,li/offset,0.2,0.20,1.00000
906,1.6/Mods/EPOE/Patches/hediffdefs_bionicsribstier2.xml,WakeUpRibQuasar,costList/BOR_QuasarAlloy,10,11,1.10000


In [5]:
# ── Cell 5: Apply Changes ────────────────────────────────────────────────────
# ⚠  Set APPLY = True to write changes to disk.

APPLY: bool = True   # ← flip to True when ready


def apply_changes(files: list, mapping: dict, *, dry_run: bool = True) -> pd.DataFrame:
    """
    Apply scaling.  Uses lxml for accurate parent-context matching, then writes
    via byte-level regex substitution to preserve original file formatting exactly.
    """
    all_rows = []

    for fpath in files:
        try:
            tree = ET.parse(str(fpath))
        except ET.XMLSyntaxError as exc:
            print(f'⚠  Parse error: {fpath.name}: {exc}')
            continue

        rel       = str(fpath.relative_to(MOD_ROOT.parent))
        file_rows = []

        for el in tree.getroot().iter():
            if not isinstance(el.tag, str):
                continue
            text = (el.text or '').strip()
            if not _is_numeric_text(text):
                continue
            parent = el.getparent()
            if parent is None:
                continue
            factor = resolve_factor(parent.tag, el.tag, mapping)
            if factor is None:
                continue
            new_text = scale_value(text, factor)
            if new_text == text:
                continue
            file_rows.append({
                'file':   rel,
                'def':    _find_def_name(el),
                'path':   f'{parent.tag}/{el.tag}',
                '_tag':   el.tag,
                'old':    text,
                'new':    new_text,
                'factor': factor,
            })

        if file_rows and not dry_run:
            raw      = fpath.read_bytes()
            decl_enc = re.search(rb'encoding=["\']([^"\']+)["\']+', raw[:100])
            enc      = decl_enc.group(1).decode() if decl_enc else 'utf-8'
            content  = raw.decode(enc)

            for row in file_rows:
                tag, old_v, new_v = row['_tag'], row['old'], row['new']
                pat     = rf'(<{re.escape(tag)}>)\s*{re.escape(old_v)}\s*(</{re.escape(tag)}>)'
                content = re.sub(pat, rf'\g<1>{new_v}\g<2>', content)

            backed_up = backup_file(fpath)
            fpath.write_bytes(content.encode(enc))
            bak_note = ' (backup created)' if backed_up else ' (backup already up to date)'
            print(f'  ✓ {rel}: {len(file_rows)} change(s){bak_note}')

        all_rows.extend(file_rows)

    cols = ['file', 'def', 'path', 'old', 'new', 'factor']
    df   = pd.DataFrame([{k: r[k] for k in cols} for r in all_rows], columns=cols) if all_rows \
           else pd.DataFrame(columns=cols)
    return df


_target_paths = [Path(f) for f in TARGET_FILES]

if not APPLY:
    print('APPLY = False — no files written.')
    print(f'Dry-run (Cell 4) shows {len(_preview_df)} pending change(s) across {len(_target_paths)} file(s).')
    print('Set APPLY = True and re-run this cell to apply.')
else:
    print(f'Applying changes to {len(_target_paths)} file(s)...\n')
    _apply_df = apply_changes(_target_paths, NODE_SCALING, dry_run=False)
    print(f'\nDone — {len(_apply_df)} change(s) written.')
    display(_apply_df)


Applying changes to 32 file(s)...

  ✓ 1.6/Common/Defs/BodyPartDefs/Tier2/BOR_CombatStatImplants.xml: 34 change(s) (backup created)
  ✓ 1.6/Common/Defs/BodyPartDefs/Tier2/BOR_StatBoostImplants.xml: 6 change(s) (backup created)
  ✓ 1.6/Common/Defs/BodyPartDefs/Tier2/BOR_StatBoostImplants_Expanded.xml: 21 change(s) (backup created)
  ✓ 1.6/Common/Defs/BodyPartDefs/Tier2/Bodyparts.xml: 32 change(s) (backup created)
  ✓ 1.6/Common/Defs/ThingDefs_Buildings/tier2/Buildings_Joy.xml: 19 change(s) (backup created)
  ✓ 1.6/Common/Defs/ThingDefs_Buildings/tier2/Buildings_Misc.xml: 18 change(s) (backup created)
  ✓ 1.6/Common/Defs/ThingDefs_Buildings/tier2/Buildings_Power.xml: 14 change(s) (backup created)
  ✓ 1.6/Common/Defs/ThingDefs_Buildings/tier2/Buildings_Production.xml: 25 change(s) (backup created)
  ✓ 1.6/Common/Defs/ThingDefs_Buildings/tier2/Buildings_Turret.xml: 14 change(s) (backup created)
  ✓ 1.6/Common/Defs/ThingDefs_Items/Tier2/Armor/BOR_StatBoostApparel.xml: 54 change(s) (backup c

,file,def,path,old,new,factor
0,1.6/Common/Defs/BodyPartDefs/Tier2/BOR_CombatStatImplants.xml,BOR_QuasarDuelflowLattice,costList/BOR_QuasarAlloy,10,11,1.10000
1,1.6/Common/Defs/BodyPartDefs/Tier2/BOR_CombatStatImplants.xml,BOR_QuasarDuelflowLattice,statBases/Mass,0.17,0.19,1.10000
2,1.6/Common/Defs/BodyPartDefs/Tier2/BOR_CombatStatImplants.xml,BOR_QuasarGraviticStrikeGovernor,costList/BOR_QuasarAlloy,10,11,1.10000
3,1.6/Common/Defs/BodyPartDefs/Tier2/BOR_CombatStatImplants.xml,BOR_QuasarGraviticStrikeGovernor,statBases/Mass,0.17,0.19,1.10000
4,1.6/Common/Defs/BodyPartDefs/Tier2/BOR_CombatStatImplants.xml,BOR_QuasarReflexOverclockSpindle,costList/BOR_QuasarAlloy,10,11,1.10000
...,...,...,...,...,...,...
903,1.6/Mods/EPOE/Patches/hediffdefs_bionicsribstier2.xml,RespirationRibQuasar,statBases/Mass,0.6,0.66,1.10000
904,1.6/Mods/EPOE/Patches/hediffdefs_bionicsribstier2.xml,WakeUpRibQuasar,statOffsets/MentalBreakThreshold,0.05,0.04,0.83334
905,1.6/Mods/EPOE/Patches/hediffdefs_bionicsribstier2.xml,WakeUpRibQuasar,li/offset,0.2,0.20,1.00000
906,1.6/Mods/EPOE/Patches/hediffdefs_bionicsribstier2.xml,WakeUpRibQuasar,costList/BOR_QuasarAlloy,10,11,1.10000


In [6]:
# ── Cell 6: Verification ─────────────────────────────────────────────────────
# Compares each current file against its backup in Scripts/backup/ to confirm
# all expected changes landed.

_ok_total, _fail_total = 0, 0

for _tf in [Path(f) for f in TARGET_FILES]:
    _bak = SCRIPT_DIR / 'backup' / _tf.relative_to(MOD_ROOT.parent)

    if not _bak.exists():
        print(f'⚠  No backup for {_tf.name} — run Cell 5 with APPLY = True first.')
        print(f'   Expected: {_bak}')
        continue

    _expected_df = preview_changes([_bak], NODE_SCALING)

    if _expected_df.empty:
        print(f'⚠  {_tf.name}: nothing expected to change (no NODE_SCALING matches).')
        continue

    _raw = _tf.read_bytes()
    _dec = re.search(rb'encoding=["\']([^"\']+)["\']+', _raw[:100])
    _cur = _raw.decode(_dec.group(1).decode() if _dec else 'utf-8')

    _ok, _fail = [], []
    for _, _row in _expected_df.iterrows():
        _tag     = _row['path'].split('/')[-1]
        _pat_new = rf'<{re.escape(_tag)}>\s*{re.escape(_row["new"])}\s*</{re.escape(_tag)}>'
        _pat_old = rf'<{re.escape(_tag)}>\s*{re.escape(_row["old"])}\s*</{re.escape(_tag)}>'
        if re.search(_pat_old, _cur):
            _fail.append(_row)
        else:
            _ok.append(_row)

    _ok_total   += len(_ok)
    _fail_total += len(_fail)

    if not _fail:
        print(f'✓ {_tf.name}: {len(_ok)} change(s) confirmed.')
    else:
        print(f'⚠  {_tf.name}: {len(_fail)} change(s) NOT applied:')
        display(pd.DataFrame(_fail)[['def', 'path', 'old', 'new']])
        if _ok:
            print(f'   ({len(_ok)} change(s) confirmed OK)')

print(f'\nTotal: {_ok_total} confirmed, {_fail_total} pending.')


✓ BOR_CombatStatImplants.xml: 34 change(s) confirmed.
✓ BOR_StatBoostImplants.xml: 6 change(s) confirmed.
⚠  BOR_StatBoostImplants_Expanded.xml: 4 change(s) NOT applied:


,def,path,old,new
1,BOR_QuasarSurgicalCortexWeave,statBases/Mass,0.18,0.20
4,BOR_QuasarInterrogatorLarynx,costList/BOR_QuasarAlloy,10,11
9,BOR_QuasarMnemonicReaderMesh,statBases/Mass,0.13,0.14
15,BOR_QuasarCircadianStabilizerMatrix,costList/BOR_QuasarAlloy,10,11


   (17 change(s) confirmed OK)
✓ Bodyparts.xml: 32 change(s) confirmed.
✓ Buildings_Joy.xml: 19 change(s) confirmed.
✓ Buildings_Misc.xml: 18 change(s) confirmed.
✓ Buildings_Power.xml: 14 change(s) confirmed.
⚠  Buildings_Production.xml: 1 change(s) NOT applied:


,def,path,old,new
10,BOR_Forge_II,li/basePowerConsumption,12500,10417


   (24 change(s) confirmed OK)
✓ Buildings_Turret.xml: 14 change(s) confirmed.
✓ BOR_StatBoostApparel.xml: 54 change(s) confirmed.
✓ apparel_armor.xml: 57 change(s) confirmed.
✓ apparel_headgear.xml: 19 change(s) confirmed.
✓ Canter.xml: 17 change(s) confirmed.
✓ Castra.xml: 8 change(s) confirmed.
✓ Sleepquitin.xml: 16 change(s) confirmed.
✓ Sunlight.xml: 16 change(s) confirmed.
✓ quasaralloy.xml: 18 change(s) confirmed.
✓ quasaralloyweave.xml: 15 change(s) confirmed.
✓ Tier2_Apparel_Belts.xml: 5 change(s) confirmed.
✓ medical.xml: 5 change(s) confirmed.
✓ MeleeUltratech.xml: 6 change(s) confirmed.
⚠  RangedUltratech.xml: 2 change(s) NOT applied:


...

   (53 change(s) confirmed OK)
✓ RangedMechanoid_Medium.xml: 42 change(s) confirmed.
✓ Races_Mechanoids_Heavy.xml: 47 change(s) confirmed.
⚠  Races_Mechanoids_Light.xml: 4 change(s) NOT applied:


,def,path,old,new
37,?,value/ArmorRating_Blunt,12,10
42,?,value/MeleeCritChance,0.04,0.03
43,?,value/MeleeParryChance,0.04,0.03
59,?,value/MeleeCritChance,0.05,0.04


   (92 change(s) confirmed OK)
⚠  Races_Mechanoids_Medium.xml: 1 change(s) NOT applied:


,def,path,old,new
43,?,value/MeleeCritChance,0.11,0.09


   (100 change(s) confirmed OK)
⚠  No backup for Recipes_MechGestator_Heavy.xml — run Cell 5 with APPLY = True first.
   Expected: /mnt/c/Program Files (x86)/Steam/steamapps/common/RimWorld/Mods/BeyondOurReach/Scripts/backup/1.6/Mods/Biotech/Patches/Tier2Mechs/Recipes/Recipes_MechGestator_Heavy.xml
⚠  No backup for Recipes_MechGestator_Light.xml — run Cell 5 with APPLY = True first.
   Expected: /mnt/c/Program Files (x86)/Steam/steamapps/common/RimWorld/Mods/BeyondOurReach/Scripts/backup/1.6/Mods/Biotech/Patches/Tier2Mechs/Recipes/Recipes_MechGestator_Light.xml
⚠  No backup for Recipes_MechGestator_Medium.xml — run Cell 5 with APPLY = True first.
   Expected: /mnt/c/Program Files (x86)/Steam/steamapps/common/RimWorld/Mods/BeyondOurReach/Scripts/backup/1.6/Mods/Biotech/Patches/Tier2Mechs/Recipes/Recipes_MechGestator_Medium.xml
✓ tier2hediffs.xml: 32 change(s) confirmed.
✓ hediffdefs_bionicsribstier2.xml: 28 change(s) confirmed.

Total: 895 confirmed, 13 pending.
